# Analisis Econometrico -- Bodega & Agroindustria Andina S.A.

Motor econometrico de la capa `04_Econometric_Analysis`, sobre datos reales/calibrados 2021-2025 (36 SKUs).
Fuentes reales usadas: BCRA (FX mayorista de referencia, BADLAR), INDEC (IPC nacional), INV (consumo domestico de vino 2024).
Metodologia: STL (estacionalidad), OLS log-log con efectos fijos + HC1 (elasticidad), Engle-Granger con chequeo ADF previo (cointegracion), SARIMA con backtest (forecast), CIP (curva Rofex teorica).

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path.cwd().parent  # 04_Econometric_Analysis/
PROJECT_ROOT = BASE.parent
sys.path.insert(0, str(BASE / "src"))

GOLD = PROJECT_ROOT / "00_Data_Engineering_ETL" / "curated_gold"
REAL = BASE / "data" / "real"
OUT = BASE / "outputs"
OUT.mkdir(exist_ok=True)

dim_productos = pd.read_csv(GOLD / "dim_productos.csv")
fact_ventas = pd.read_csv(GOLD / "fact_ventas_reales.csv", parse_dates=["Fecha"])
fx_real = pd.read_csv(REAL / "fx_mensual_2021_2025.csv", parse_dates=["date"], index_col="date")["value"]
ipc_real = pd.read_csv(REAL / "ipc_mensual_2021_2025.csv", parse_dates=["date"], index_col="date")["value"]

print(f"fact_ventas_reales: {len(fact_ventas):,} filas | {fact_ventas['Fecha'].min().date()} a {fact_ventas['Fecha'].max().date()} | {dim_productos.shape[0]} SKUs")

fact_ventas_reales: 29,207 filas | 2021-01-01 a 2025-12-27 | 36 SKUs


## 1. Estacionalidad real (STL)

Serie: consumo domestico agregado mensual 2021-2025 (excluye canal exportacion). El indice estacional inyectado en los datos se calibro contra consumo real INV mercado interno 2024 (unico anio con granularidad mensual publica). Recuperarlo via STL valida la metodologia, no es evidencia de negocio independiente de la calibracion.

In [2]:
from estacionalidad_y_forecast import ejecutar_tarea1_estacionalidad, ejecutar_tarea2_forecast

est = json.load(open(OUT / "estacionalidad_resultados.json", encoding="utf-8"))
print(f"Correlacion STL vs indice real INV 2024: {est['correlacion_con_indice_real']:.4f}")
print(est["nota_metodologica"])
pd.DataFrame(est["tabla_mes_a_mes"])

Correlacion STL vs indice real INV 2024: 0.9854
El indice real INV2024 fue la referencia usada para calibrar la estacionalidad sintetica de fact_ventas_reales.csv. Esta comparacion valida que STL recupera el patron inyectado (validacion de metodo), no es evidencia de negocio independiente.


,mes,indice_estacional_recuperado_stl,indice_estacional_real_inv2024,diferencia_absoluta
0,1,0.828374,0.7851,0.043274
1,2,0.803057,0.7864,0.016657
2,3,0.835335,0.8533,0.017965
3,4,0.840803,0.8485,0.007697
4,5,1.081084,1.0485,0.032584
5,6,0.845543,0.9153,0.069757
6,7,1.179937,1.1862,0.006263
7,8,1.300984,1.2806,0.020384
8,9,1.107250,1.1087,0.001450
9,10,1.135698,1.1388,0.003102


## 2. Forecast de demanda (SARIMA, backtest 12 meses)

Entrenado en 2021-2024 (48 meses), backtesteado contra 2025 (12 meses fuera de muestra). Seleccion de orden por AIC en train dentro de una grilla acotada.

In [3]:
fcst = json.load(open(OUT / "forecast_resultados.json", encoding="utf-8"))
print(f"Modelo elegido (AIC={fcst['aic_train']:.2f}): {fcst['orden_modelo']}")
print(f"MAPE test (12m 2025): {fcst['MAPE']:.2f}%  |  RMSE: {fcst['RMSE']:.2f}")
pd.DataFrame({"real": fcst["real_12m"], "forecast": fcst["forecast_12m"]})

Modelo elegido (AIC=404.61): {'order': [0, 1, 1], 'seasonal_order': [0, 1, 1, 12]}
MAPE test (12m 2025): 4.77%  |  RMSE: 3176.27


,real,forecast
2025-01-01,45643.0,39449.92
2025-02-01,40886.0,41326.99
2025-03-01,44933.0,43678.53
2025-04-01,43426.0,44159.05
2025-05-01,51072.0,55463.72
2025-06-01,44783.0,42904.23
2025-07-01,55763.0,61809.03
2025-08-01,66001.0,64533.67
2025-09-01,58548.0,54616.55
2025-10-01,58271.0,56879.42


## 3. Elasticidad-precio (panel SKU x mes, canales domesticos)

OLS log-log con efectos fijos de SKU y mes calendario, errores robustos HC1. Beta = elasticidad-precio propia.

**Resultado honesto**: el orden economico esperado (Entrada/Granel mas elastico que Icono) **no se cumple** en 4 de 5 segmentos -- salen con beta positivo y no significativo. Solo Gran Reserva/Icono tiene un beta negativo, grande y altamente significativo (p<0.001). Diagnostico: el precio domestico de cada SKU se construyo como IPC-mensual x deriva-real-de-segmento (comun a todos los SKU del segmento) mas 2% de ruido idiosincratico -- para los segmentos con deriva real chica (Entrada -1.5%/anio, Espumante -0.5%/anio) la variacion de precio *relativa* entre SKUs y en el tiempo es casi toda ruido, insuficiente para identificar beta contra los efectos fijos de mes-calendario (que no absorben tendencias plurianuales). Icono, con la deriva real mas grande (+2%/anio) y el mayor rango de precio en USD, es el unico segmento con "tratamiento" (variacion de precio) suficientemente grande para separarse del ruido. Es una leccion real de identificacion causal (variacion de precio endógena/insuficiente), no un resultado forzado -- coherente con lo que se ve en la practica: sin experimentos de precio o instrumentos, la elasticidad de productos con pricing muy uniforme es dificil de identificar limpiamente.

In [4]:
elas = json.load(open(OUT / "elasticidad_resultados.json", encoding="utf-8"))
df_elas = pd.DataFrame(elas["resultados_por_segmento"]).T
print(elas["chequeo_sentido_economico"])
df_elas

OBSERVACION -- el orden economico esperado NO se cumple en los datos: segmentos de necesidad/commodity ({'Entrada': 0.0030394422563185843, 'Granel / Masivo': 0.024212411141490114}) no resultaron mas elasticos que Gran Reserva/Icono ({'Gran Reserva / Icono': -0.11144816516413378}). Se reporta el resultado real, sin forzarlo.


,beta,std_err,p_value,r2,n_obs
GLOBAL,-0.016127,0.008081,0.045975,0.461875,2159.0
Entrada,0.003039,0.012044,0.800765,0.424313,780.0
Espumante,0.017015,0.023234,0.463955,0.289497,180.0
Gran Reserva / Icono,-0.111448,0.029622,0.000168,0.165644,240.0
Granel / Masivo,0.024212,0.021753,0.265683,0.408663,180.0
Reserva,-0.019958,0.014185,0.159441,0.359336,779.0


## 4. Cointegracion (Engle-Granger con chequeo ADF previo)

Test A: precio de exportacion ARS vs. FX real BCRA (pass-through cambiario). Test B: precio domestico nominal vs. IPC real INDEC (pass-through inflacionario).

**Resultado honesto**: ninguno de los dos tests es valido en sentido estricto -- ambas series de precio nominal resultaron I(2)+ (necesitan dos diferencias para ser estacionarias), no I(1) como exige Engle-Granger. Es plausible y economicamente razonable: con inflacion acelerando (no solo alta, sino con tasa de crecimiento creciente) en 2021-2025, los niveles nominales en pesos pueden requerir doble diferenciacion. La funcion igual devuelve el estadistico a titulo informativo, marcado explicitamente como no valido -- se reporta el resultado real, no se descarta el chequeo de requisitos para forzar una conclusion.

In [5]:
coint = json.load(open(OUT / "cointegracion_resultados.json", encoding="utf-8"))
for nombre, r in coint.items():
    print(f"--- {nombre} ---")
    print(f"  y: {r['orden_y']}  |  x: {r['orden_x']}  |  valido: {r['es_valido']}")
    print(f"  {r['conclusion']}\
")

--- test_A_pass_through_cambiario ---
  y: I(2)+  |  x: I(1)  |  valido: False
  Test no valido en sentido estricto: y es I(2)+, x es I(1). Engle-Granger requiere ambas series I(1). Resultado reportado a titulo informativo unicamente.
--- test_B_pass_through_inflacionario ---
  y: I(2)+  |  x: I(2)+  |  valido: False
  Test no valido en sentido estricto: y es I(2)+, x es I(2)+. Engle-Granger requiere ambas series I(1). Resultado reportado a titulo informativo unicamente.


## 5. Curva Rofex teorica (CIP -- Covered Interest Rate Parity)

Sin API publica de futuros Matba-Rofex, se reconstruye el forward teorico a 90 dias por no-arbitraje: `F = S * (1+i_dom*d/365) / (1+i_ext*d/365)`. Spot: FX real BCRA (mayorista de referencia). Tasa domestica: BADLAR bancos privados real BCRA (`idVariable=7`, serie diaria completa 2021-2025, sin proxy). Tasa externa: 4.5% TNA (proxy Fed Funds/T-Bill corto plazo, no hay serie argentina equivalente en USD).

In [6]:
curva_rofex = pd.read_csv(OUT / "curva_rofex_teorica_2021_2025.csv", parse_dates=["fecha"])
corr_spot_fwd = curva_rofex["spot_ars_usd"].corr(curva_rofex["forward_teorico_90d"])
print(f"Correlacion spot vs. forward teorico 90d: {corr_spot_fwd:.4f}")
print(f"TNA implicita del forward -- min: {curva_rofex['tna_implicita_forward'].min():.1%}  "
      f"max: {curva_rofex['tna_implicita_forward'].max():.1%}  "
      f"promedio: {curva_rofex['tna_implicita_forward'].mean():.1%}")
curva_rofex.tail(6)

Correlacion spot vs. forward teorico 90d: 0.9972
TNA implicita del forward -- min: 21.7%  max: 123.5%  promedio: 49.6%


,fecha,spot_ars_usd,tna_domestica_usada,tna_externa_usada,forward_teorico_90d,tna_implicita_forward
54,2025-07-01,1267.022732,0.324972,0.045,1353.530739,0.276900
55,2025-08-01,1329.537495,0.480312,0.045,1470.680230,0.430535
56,2025-09-01,1399.897736,0.484290,0.045,1549.867922,0.434469
57,2025-10-01,1432.022727,0.461676,0.045,1577.537035,0.412103
58,2025-11-01,1427.617647,0.320257,0.045,1523.448991,0.272236
59,2025-12-01,1447.837721,0.264605,0.045,1525.376581,0.217195


## 6. Sintesis

| Analisis | Resultado | Lectura |
|---|---|---|
| Estacionalidad (STL) | corr. 0.985 vs. indice real INV 2024 | Metodologia valida; el patron real (invierno alto, verano bajo) queda correctamente recuperado |
| Forecast (SARIMA, backtest 12m) | MAPE 4.77%, RMSE 3176 | Forecast solido para planificacion de demanda agregada mensual |
| Elasticidad-precio | Solo Icono significativo (beta=-0.11, p<0.001); resto no identificado | Limite real de identificacion con pricing poco idiosincratico -- hallazgo honesto, no forzado |
| Cointegracion (FX y IPC) | Ninguna valida (series I(2)+) | Consistente con inflacion acelerando, no solo alta, en 2021-2025 |
| Curva Rofex (CIP) | corr. spot-forward 0.997, TNA implicita 22%-124% | Anclaje real (BADLAR + FX BCRA), sin inventar forma de curva |

Todas las fuentes de calibracion (FX, BADLAR, IPC, estacionalidad de consumo) son datos publicos reales (BCRA, INDEC, INV/magyp), no inventados. Los datos de venta subyacentes (36 SKUs, 2021-2025) son sinteticos pero generados para ser consistentes con esas series reales -- no existe un dataset transaccional real disponible para esta bodega ficticia, asi que la honestidad metodologica esta en la calibracion, no en pretender que las transacciones mismas son reales.